In [ ]:
!git clone -b feature/hebrew-fine-tune https://github.com/thewh1teagle/mixer-tts-pytorch.git
%cd mixer-tts-pytorch
!apt-get install -y espeak-ng

In [ ]:
!uv sync

In [ ]:
!mkdir -p data
!wget -O data/arabic-small-tts-dataset.zip https://github.com/thewh1teagle/mixer-tts-pytorch/releases/download/model-files-v1.0/arabic-small-tts-dataset.zip
!unzip -o data/arabic-small-tts-dataset.zip -d data/

In [ ]:
!uv pip install "phoonnx[ar] @ git+https://github.com/TigreGotico/phoonnx.git"

import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.as_posix() not in sys.path:
    sys.path.insert(0, repo_root.as_posix())

from export_onnx import ARABIC_IPA_REPLACEMENTS
from phoonnx.config import Alphabet
from phoonnx.phonemizers.ar import MantoqPhonemizer
from models.symbols import symbols_to_id

dataset_root = Path("data/arabic-small-tts-dataset")
audio_dir = dataset_root / "wavs"
metadata_in = dataset_root / "metadata.csv"
metadata_out = dataset_root / "metadata_phonemes.csv"

phonemizer = MantoqPhonemizer(alphabet=Alphabet.IPA)


def normalize_arabic_ipa(ipa: str) -> str:
    for source, target in ARABIC_IPA_REPLACEMENTS.items():
        ipa = ipa.replace(source, target)
    return ipa


rows = []
skipped_malformed = 0
skipped_missing_audio = 0
missing_symbols = {}
for line_no, line in enumerate(metadata_in.read_text(encoding="utf-8").splitlines(), 1):
    if not line.strip():
        continue
    parts = line.split("|", 2)
    if len(parts) < 2:
        skipped_malformed += 1
        continue
    audio_id, text = parts[0], parts[1]
    if not (audio_dir / f"{audio_id}.wav").exists():
        skipped_missing_audio += 1
        continue

    # Do not call add_diacritics(); this dataset is already vocalized.
    ipa = normalize_arabic_ipa(phonemizer.phonemize_string(text, "ar")).strip()
    rows.append(f"{audio_id}|{text}|{ipa}")

    for ch in f" {ipa} ":
        if ch not in symbols_to_id:
            missing_symbols[ch] = missing_symbols.get(ch, 0) + 1

metadata_out.write_text("\n".join(rows) + "\n", encoding="utf-8")
print(f"wrote {len(rows)} rows to {metadata_out}")
print(f"skipped_malformed={skipped_malformed} skipped_missing_audio={skipped_missing_audio}")
print("missing_symbols:", missing_symbols)
print("sample:", rows[0] if rows else "")

In [ ]:
!uv run python download_files.py

In [ ]:
from pathlib import Path
import yaml

base_config = Path("configs/arabic-tts-v1-80.yaml")
small_config = Path("configs/arabic-small-tts.yaml")

config = yaml.safe_load(base_config.read_text(encoding="utf-8"))
config.update({
    "log_dir": "logs/arabic-small-tts",
    "checkpoint_dir": "checkpoints/arabic-small-tts",
    "train_audio_dir": "./data/arabic-small-tts-dataset/wavs",
    "train_labels": "./data/arabic-small-tts-dataset/metadata_phonemes.csv",
    "pitch_dir": "./data/arabic-small-tts-dataset/pitch",
    "eval_fraction": 0.2,
    "eval_interval": 100,
    "save_interval": 100,
    "print_interval": 1,
    "num_workers": 0,
})

small_config.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
print(small_config.read_text(encoding="utf-8"))

In [ ]:
!uv pip install git+https://github.com/thewh1teagle/rmvpe-onnx
!mkdir -p data
!wget -nc -O data/rmvpe.onnx https://github.com/thewh1teagle/rmvpe-onnx/releases/download/model-files-v1.0/rmvpe.onnx

!uv run python extract_pitch_dataset.py \
  --audio-dir ./data/arabic-small-tts-dataset/wavs \
  --metadata ./data/arabic-small-tts-dataset/metadata_phonemes.csv \
  --pitch-dir ./data/arabic-small-tts-dataset/pitch \
  --rmvpe-model ./data/rmvpe.onnx \
  --sample-rate 22050 \
  --overwrite

In [ ]:
!uv run python train_ft.py --config ./configs/arabic-small-tts.yaml --max-steps 501 --eval-max-batches 1

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.as_posix() not in sys.path:
    sys.path.insert(0, repo_root.as_posix())

import soundfile as sf
import torch
from IPython.display import Audio, display
from vocos import Vocos

from models import MixerTTSModel
from models.symbols import symbols_to_id

device = "cuda:0" if torch.cuda.is_available() else "cpu"
checkpoint_path = Path("checkpoints/arabic-small-tts/last.pth")
metadata_path = Path("data/arabic-small-tts-dataset/metadata_phonemes.csv")
output_path = Path("outputs/arabic-small-sample.wav")
output_path.parent.mkdir(parents=True, exist_ok=True)

audio_id, text, ipa = metadata_path.read_text(encoding="utf-8").splitlines()[0].split("|", 2)
ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
model = MixerTTSModel(**ckpt["net_config"]).to(device).eval()
model.load_state_dict(ckpt["model"], strict=False)

token_ids = torch.LongTensor([symbols_to_id[ch] for ch in f" {ipa} " if ch in symbols_to_id]).to(device)
token_lens = torch.LongTensor([token_ids.numel()]).to(device)

with torch.inference_mode():
    mel = model.infer(token_ids[None], token_lens)
    vocos = Vocos.from_pretrained("BSC-LT/vocos-mel-22khz").to(device).eval()
    wav = vocos.decode(mel.transpose(1, 2))[0].detach().cpu()
    wav = wav / wav.abs().max().clamp_min(1e-6)

sf.write(output_path, wav.numpy(), 22050, subtype="PCM_16")
print(f"id={audio_id}")
print(f"text={text}")
print(f"ipa={ipa}")
print(f"checkpoint_iter={ckpt.get('iter')}")
print(output_path)
display(Audio(output_path.as_posix(), rate=22050))

In [ ]:
!mkdir -p outputs
!uv run python export_onnx.py \
  --checkpoint ./checkpoints/arabic-small-tts/last.pth \
  --output ./outputs/arabic-small-tts-vocos.onnx \
  --with-vocoder \
  --vocoder-kind vocos \
  --vocoder BSC-LT/vocos-mel-22khz \
  --no-int8

In [ ]:
import sys
from pathlib import Path

from IPython.display import Audio, display

repo_root = Path.cwd()
onnx_wrapper_src = repo_root / "mixer-tts-onnx" / "src"
if onnx_wrapper_src.as_posix() not in sys.path:
    sys.path.insert(0, onnx_wrapper_src.as_posix())

from mixer_tts_onnx import MixerTTS

model_path = Path("outputs/arabic-small-tts-vocos.onnx")
metadata_path = Path("data/arabic-small-tts-dataset/metadata_phonemes.csv")
output_path = Path("outputs/arabic-small-onnx-sample.wav")

audio_id, text, ipa = metadata_path.read_text(encoding="utf-8").splitlines()[0].split("|", 2)
tts = MixerTTS(model_path)
tts.create(ipa, is_phonemes=True, output_path=output_path, speed=1.0)

print(f"id={audio_id}")
print(f"text={text}")
print(f"ipa={ipa}")
print(output_path)
display(Audio(output_path.as_posix(), rate=tts.sample_rate))

In [ ]:
!uv run python train_ft.py --config ./configs/arabic-small-tts.yaml